In [5]:
import pandas as pd

# Load the NEW large database prepared by data_preparation.py (with wider area and more points!)
dfc = pd.read_csv(r"C:\Users\User\OneDrive\Documents\GitHub\V-2-V--logistics\central agent\fixed_database_large.csv")

# The rest of the notebook expects `dfc_propre`
dfc_propre = dfc.copy()

print("Colonnes:", list(dfc.columns))
print("Nb lignes:", len(dfc))
dfc.head()

Colonnes: ['Reference', 'Expéditeur', 'Adresse destinataire', 'Action', 'Contenu', 'Poids', 'Date de création', 'Statut', 'Gouvernorat', 'Prix Total', 'lat', 'lon', 'ExpÃ©diteur', 'Date de crÃ©ation', 'Frais_Livraison_TND', 'Priorite', 'Mode_Paiement', 'Type_Client']
Nb lignes: 48


,Reference,Expéditeur,Adresse destinataire,Action,Contenu,Poids,Date de création,Statut,Gouvernorat,Prix Total,lat,lon,ExpÃ©diteur,Date de crÃ©ation,Frais_Livraison_TND,Priorite,Mode_Paiement,Type_Client
0,952897325079,NaN,Zone carrefour el marsa,NaN,VÃªtements,0.5,NaN,En attente,Grand Tunis,NaN,36.630960,10.079507,Boutique,2025-11-01T08:04:09+01:00,7.75,Express,Paiement Ã la livraison,B2C (Particulier)
1,842619805665,NaN,Zone zahrouni,NaN,Chaussures,1.3,NaN,En attente,Grand Tunis,NaN,36.891938,10.123793,Boutique,2025-11-01T08:16:34+01:00,8.95,Standard,Paiement Ã la livraison,B2C (Particulier)
2,467572401819,NaN,Zone ezzahra,NaN,Accessoires,0.3,NaN,En attente,Grand Tunis,NaN,36.850436,10.225890,Boutique,2025-11-01T08:21:41+01:00,7.45,Express,Paiement Ã la livraison,B2C (Particulier)
3,815684550792,NaN,Zone jardins d'el menzah 1,NaN,VÃªtements,0.5,NaN,En cours,Grand Tunis,NaN,36.635900,9.918795,Boutique,2025-11-01T08:35:06+01:00,7.75,Express,Paiement Ã la livraison,B2C (Particulier)
4,242654144774,NaN,Zone Kram,NaN,Accessoires,0.3,NaN,PrÃªt pour livraison,Grand Tunis,NaN,36.750983,10.223137,Boutique,2025-11-01T08:36:22+01:00,7.45,Express,Paiement Ã la livraison,B2C (Particulier)


In [6]:
import json
import html
from pathlib import Path

# Optionnel: si tu as une vraie clÃ© Google Maps JavaScript API, mets-la ici.
# Sinon, le notebook gÃ©nÃ¨re une solution Google Maps SANS clÃ© (KML + liens + page HTML).
GOOGLE_MAPS_API_KEY = ""  # ex: "AIza...." (laisser vide si tu n'as pas de clÃ©)

# On utilise dfc_propre (la grosse base augmentÃ©e) au lieu de df_map (seulement les originaux)
if "dfc_propre" not in globals() or dfc_propre is None or dfc_propre.empty:
    raise ValueError("dfc_propre is empty. Veuillez gÃ©nÃ©rer les donnÃ©es supplÃ©mentaires en premier.")

points = dfc_propre.dropna(subset=["lat", "lon"]).copy()

markers = []
for _, row in points.iterrows():
    ref = "" if pd.isna(row.get("Reference")) else str(row.get("Reference"))
    adr = "" if pd.isna(row.get("Adresse destinataire")) else str(row.get("Adresse destinataire"))
    statut = "" if pd.isna(row.get("Statut")) else str(row.get("Statut"))
    
    # On peut aussi ajouter les nouvelles colonnes dans l'affichage !
    frais = f"{row.get('Frais_Livraison_TND', 'N/A')} TND"
    priorite = "" if pd.isna(row.get("Priorite")) else str(row.get("Priorite"))

    title = f"{ref} - {adr}".strip(" -")
    info_html = (
        f"<div>"
        f"<b>RÃ©fÃ©rence:</b> {html.escape(ref)}<br/>"
        f"<b>Adresse:</b> {html.escape(adr)}<br/>"
        f"<b>Frais:</b> {html.escape(frais)}<br/>"
        f"<b>PrioritÃ©:</b> {html.escape(priorite)}<br/>"
        f"<b>Statut:</b> {html.escape(statut)}"
        f"</div>"
    )

    markers.append(
        {
            "lat": float(row["lat"]),
            "lng": float(row["lon"]),
            "title": title,
            "infoHtml": info_html,
        }
    )

# --- Ajout du point de l'entrepÃ´t ---
markers.append(
    {
        "lat": 36.82857,
        "lng": 10.20616,
        "title": "EntrepÃ´t Principal",
        "infoHtml": (
            "<div>"
            "<b>EntrepÃ´t Principal</b><br/>"
            "Point de dÃ©part des livraisons"
            "</div>"
        ),
    }
)

# ----------------------------
# 1) Export KML (Google My Maps)
# ----------------------------
# Google My Maps accepte l'import KML (sans API key).

kml_path = Path("livraisons_mymaps.kml")

placemarks = []
for m in markers:
    name = html.escape(m["title"] or "Destination")
    desc = m["infoHtml"]
    lon = m["lng"]
    lat = m["lat"]
    placemarks.append(
        "\n".join(
            [
                "<Placemark>",
                f"  <name>{name}</name>",
                f"  <description><![CDATA[{desc}]]></description>",
                "  <Point>",
                f"    <coordinates>{lon},{lat},0</coordinates>",
                "  </Point>",
                "</Placemark>",
            ]
        )
    )

kml_doc = "\n".join(
    [
        "<?xml version=\"1.0\" encoding=\"UTF-8\"?>",
        "<kml xmlns=\"http://www.opengis.net/kml/2.2\">",
        "<Document>",
        "  <name>Livraisons</name>",
        *placemarks,
        "</Document>",
        "</kml>",
        "",
    ]
)

kml_path.write_text(kml_doc, encoding="utf-8")

# ------------------------------------
# 2) HTML avec liens Google Maps (sans clÃ©)
# ------------------------------------
# Chaque lien ouvre un point directement dans Google Maps.

links_path = Path("google_maps_links.html")

rows = []
for m in markers:
    q = f"{m['lat']},{m['lng']}"
    url = f"https://www.google.com/maps/search/?api=1&query={q}"
    label = m["title"] or q
    rows.append(
        f"<li><a href=\"{url}\" target=\"_blank\" rel=\"noopener\">{html.escape(label)}</a></li>"
    )

links_html = "\n".join(
    [
        "<!doctype html>",
        "<html>",
        "<head><meta charset=\"utf-8\"><title>Livraisons - Liens Google Maps</title></head>",
        "<body>",
        "<h2>Destinations (ouvrir dans Google Maps)</h2>",
        "<b>Total points:</b> " + str(len(markers)),
        "<ol>",
        *rows,
        "</ol>",
        "</body>",
        "</html>",
        "",
    ]
)

links_path.write_text(links_html, encoding="utf-8")

# -------------------------------------------------
# 3) Page principale: avec clÃ© -> carte interactive, sans clÃ© -> page d'instructions
# -------------------------------------------------

main_path = Path("google_maps_livraisons.html")

if GOOGLE_MAPS_API_KEY and "PASTE_YOUR_API_KEY_HERE" not in GOOGLE_MAPS_API_KEY:
    center_lat = float(points["lat"].mean())
    center_lng = float(points["lon"].mean())

    html_doc = f"""<!doctype html>
<html>
  <head>
    <meta charset=\"utf-8\" />
    <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\" />
    <title>Livraisons - Google Maps</title>
    <style>
      html, body, #map {{ height: 100%; margin: 0; padding: 0; }}
    </style>
  </head>
  <body>
    <div id=\"map\"></div>

    <script>
      const markers = {json.dumps(markers, ensure_ascii=False)};

      function initMap() {{
        const center = {{ lat: {center_lat}, lng: {center_lng} }};
        const map = new google.maps.Map(document.getElementById('map'), {{
          zoom: 11,
          center
        }});

        const bounds = new google.maps.LatLngBounds();
        const infoWindow = new google.maps.InfoWindow();

        for (const m of markers) {{
          const pos = new google.maps.LatLng(m.lat, m.lng);
          bounds.extend(pos);

          const marker = new google.maps.Marker({{
            position: pos,
            map,
            title: m.title
          }});

          marker.addListener('click', () => {{
            infoWindow.setContent(m.infoHtml);
            infoWindow.open(map, marker);
          }});
        }}

        if (markers.length > 1) {{
          map.fitBounds(bounds);
        }}
      }}
    </script>

    <script async defer
      src=\"https://maps.googleapis.com/maps/api/js?key={GOOGLE_MAPS_API_KEY}&callback=initMap\">
    </script>
  </body>
</html>
"""

    main_path.write_text(html_doc, encoding="utf-8")
    print(f"Saved (JS API map): {main_path.resolve()}")
else:
    # IMPORTANT: sans clÃ©, on ne charge PAS l'API Google Maps JS (sinon InvalidKeyMapError)
    instructions = "\n".join(
        [
            "<!doctype html>",
            "<html>",
            "<head><meta charset=\"utf-8\"><title>Livraisons - Google Maps</title></head>",
            "<body>",
            "<h2>Livraisons (Google Maps) â€” sans clÃ© API</h2>",
            "<p>La carte Google Maps interactive nÃ©cessite une clÃ© API valide. Pour afficher tes " + str(len(markers)) + " points sur Google Maps <b>sans clÃ©</b>:</p>",
            "<ol>",
            "  <li>Va sur <a href=\"https://www.google.com/mymaps\" target=\"_blank\" rel=\"noopener\">Google My Maps</a></li>",
            "  <li>CrÃ©e une nouvelle carte</li>",
            "  <li>Clique sur <b>Importer</b> et choisis le fichier <b>livraisons_mymaps.kml</b></li>",
            "</ol>",
            "<p>Ou ouvre directement chaque point via la liste de liens:</p>",
            "<p><a href=\"google_maps_links.html\">Ouvrir google_maps_links.html</a></p>",
            "</body>",
            "</html>",
            "",
        ]
    )
    main_path.write_text(instructions, encoding="utf-8")
    print(f"Saved (no-key instructions page): {main_path.resolve()}")

print(f"Saved (My Maps import with {len(markers)} points) : {kml_path.resolve()}")
print(f"Saved (Links): {links_path.resolve()}")

Saved (no-key instructions page): C:\Users\User\OneDrive\Documents\GitHub\V-2-V--logistics\central agent\google_maps_livraisons.html
Saved (My Maps import with 49 points) : C:\Users\User\OneDrive\Documents\GitHub\V-2-V--logistics\central agent\livraisons_mymaps.kml
Saved (Links): C:\Users\User\OneDrive\Documents\GitHub\V-2-V--logistics\central agent\google_maps_links.html


In [7]:
%pip install ortools

Note: you may need to restart the kernel to use updated packages.


In [8]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import folium
from geopy.distance import geodesic

# Cordonnées de l'entrepôt
ENTREPOT = (36.82857, 10.20616)

# --- 1. PRÉPARATION DES DONNÉES ---
tous_les_points = [ENTREPOT] + list(zip(dfc_propre['lat'], dfc_propre['lon']))

def create_distance_matrix(points):
    """Crée une matrice de distance de chaque point vers tous les autres"""
    matrix = []
    for i in range(len(points)):
        row = []
        for j in range(len(points)):
            if i == j:
                row.append(0)
            else:                     
                dist_meters = int(geodesic(points[i], points[j]).meters)
                row.append(dist_meters)
        matrix.append(row)
    return matrix

print("Calcul de la matrice de distance en cours...")
distance_matrix = create_distance_matrix(tous_les_points)

# Paramètres : On définit le maximum de véhicules à 5 pour laisser l'algo répartir sans bloquer.
MAX_VEHICULES = 5
data = {
    'distance_matrix': distance_matrix,
    'num_vehicles': MAX_VEHICULES, 
    'depot': 0                  
}

# --- 2. INITIALISATION DU SOLVER OR-TOOLS ---
manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']), data['num_vehicles'], data['depot'])
routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data['distance_matrix'][from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)


# *** INTELLIGENCE DU SYSTEME : MINIMISER LE NOMBRE DE VÉHICULES ET LE CARBURANT ***

# 1. On définit un Coût Fixe par Véhicule (ex: 25 000). 
# L'algo essaiera donc de placer les colis dans le moins de véhicules possibles.
COUT_FIXE_VEHICULE = 25000 
routing.SetFixedCostOfAllVehicles(COUT_FIXE_VEHICULE)

# 2. Une dimension de distance pour garantir qu'un véhicule ne fasse pas 
# trop de kilomètres. Comme les adresses sont très éparpillées sur 120km,
# on fixe la limite à 150 km. (Pour éviter "aucune solution trouvée")
dimension_name = 'Distance'
routing.AddDimension(
    transit_callback_index,
    0,        
    150000,   # Capacité max : 150 000 mètres (150 km) par véhicule
    True,     
    dimension_name)

distance_dimension = routing.GetDimensionOrDie(dimension_name)

# --- 3. RECHERCHE DE LA SOLUTION ---
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
search_parameters.time_limit.seconds = 5 

print("Recherche de la meilleure solution optimisée...")
solution = routing.SolveWithParameters(search_parameters)


# --- 4. AFFICHAGE DES RÉSULTATS ---
if solution:
    map_ortools = folium.Map(location=[36.8065, 10.1815], zoom_start=11)
    
    folium.Marker(
        location=ENTREPOT,
        popup="<b>Entrepôt Principal (Départ)</b>",
        icon=folium.Icon(color='red', icon='industry', prefix='fa')
    ).add_to(map_ortools)

    colors_ortools = ['blue', 'green', 'purple', 'orange', 'darkred']
    
    print("\n=== PLAN DE TOURNÉES OPTIMISÉ (Carburant & Véhicules minimum) ===")
    
    vehicules_utilises = 0
    distance_totale = 0

    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route_coords = []
        distance_trajet = 0
        ordre_livraison = 1
        
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_coords.append(tous_les_points[node_index])
            
            if node_index != 0:
                folium.Marker(
                    location=tous_les_points[node_index],
                    popup=f"<b>Chauffeur {vehicules_utilises+1}</b><br>Arrêt n°{ordre_livraison}",
                    icon=folium.Icon(color=colors_ortools[vehicules_utilises % len(colors_ortools)], icon='info-sign')
                ).add_to(map_ortools)
                ordre_livraison += 1
                
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            distance_trajet += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
            
        route_coords.append(tous_les_points[manager.IndexToNode(index)])
        
        if len(route_coords) > 2:
            vehicules_utilises += 1
            distance_totale += distance_trajet
            print(f"Véhicule {vehicules_utilises} : {len(route_coords)-2} colis | Distance : {distance_trajet/1000:.2f} km")
            
            folium.PolyLine(
                route_coords,
                color=colors_ortools[vehicules_utilises % len(colors_ortools)],
                weight=4,
                opacity=0.8,
                tooltip=f"Tournée Chauffeur {vehicules_utilises} ({distance_trajet/1000:.2f} km)"
            ).add_to(map_ortools)
            
    print(f"\nRésumé : {vehicules_utilises} véhicule(s) activé(s) sur {MAX_VEHICULES} disponibles.")
    print(f"Distance cumulée de l'ensemble de la flotte : {distance_totale/1000:.2f} km")
    display(map_ortools)
else:
    print("Aucune solution trouvée par le solver (tentez d'augmenter le nombre de véhicules ou la limte de kilomètres).")

Calcul de la matrice de distance en cours...
Recherche de la meilleure solution optimisée...

=== PLAN DE TOURNÉES OPTIMISÉ (Carburant & Véhicules minimum) ===
Véhicule 1 : 27 colis | Distance : 169.99 km
Véhicule 2 : 21 colis | Distance : 146.01 km

Résumé : 2 véhicule(s) activé(s) sur 5 disponibles.
Distance cumulée de l'ensemble de la flotte : 316.00 km


In [9]:
# --- GÉNÉRATION D'UNE URL GOOGLE MAPS DIRECTIONS ---

if solution:
    print("=== LIENS GOOGLE MAPS POUR LES CHAUFFEURS ===\n")
    
    vehicules_utilises = 0
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route_coords = []
        
        # Récupérer l'itinéraire de ce chauffeur
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_coords.append(tous_les_points[node_index])
            index = solution.Value(routing.NextVar(index))
            
        # Ajouter le retour à l'entrepôt
        route_coords.append(tous_les_points[manager.IndexToNode(index)])
        
        # S'il n'y a que l'entrepôt (départ et arrivée), le chauffeur ne fait rien
        if len(route_coords) <= 2:
            continue
            
        vehicules_utilises += 1
        # Construction de l'URL Google Maps
        # Format: https://www.google.com/maps/dir/Depart/Etape1/Etape2/.../Arrivee
        
        url_parts = ["https://www.google.com/maps/dir"]
        
        for lat, lon in route_coords:
            url_parts.append(f"{lat},{lon}")
            
        # On assemble l'URL complète
        google_maps_url = "/".join(url_parts)
        
        print(f"Véhicule {vehicules_utilises} ({len(route_coords)-2} livraisons) :")
        print(f"Cliquez ici pour ouvrir le GPS : {google_maps_url}\n")
else:
    print("Aucun itinéraire disponible à exporter.")

=== LIENS GOOGLE MAPS POUR LES CHAUFFEURS ===

Véhicule 1 (27 livraisons) :
Cliquez ici pour ouvrir le GPS : https://www.google.com/maps/dir/36.82857,10.20616/36.8341704,10.194323/36.88479198756381,10.149848247932702/36.89193768186935,10.123792857555395/36.870615751499685,10.07756394625929/36.877417985245344,10.037868816641742/36.89410834031765,10.04898703673507/36.95659934226164,10.13001797876793/36.94228905486141,10.064010643898524/36.94585933185392,9.964241407057964/36.92335449965161,9.950060013956724/36.813761188766,9.978313558581684/36.73656237701236,9.956668593380902/36.69753604462167,9.913063774158225/36.55258063938274,9.863736239410798/36.55727655737609,9.880462068409244/36.63590047370475,9.91879508384335/36.64627225776566,9.94667091579683/36.713887514545256,10.100909039058994/36.76413,10.1127187/36.7686795,10.1135873/36.7922783,10.1078508/36.79935860297934,10.120430277731616/36.79917937424554,10.129838611755371/36.81730676242966,10.119789596992511/36.8372507,10.1402003/36.8353

In [ ]:
# --- SIMULATION : INSERTION D'UNE NOUVELLE COMMANDE EN TEMPS RÉEL ---
from geopy.distance import geodesic
import folium

# Simulation d'une nouvelle demande de Pick-up & Delivery
# Coordonnées arbitraires autour de Tunis
new_pickup = (36.8400, 10.2200)   # Point de ramassage
new_dropoff = (36.8600, 10.2500)  # Point de livraison (peut être l'entrepôt si c'est un retour)

print("=== NOUVELLE DEMANDE EN TEMPS RÉEL ===")
print(f"Pickup      : {new_pickup}")
print(f"Destination : {new_dropoff}\n")

# 1. Extraire les itinéraires actuels du solver
current_routes = {}
if 'solution' in globals() and solution:
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route = []
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route.append(tous_les_points[node_index])
            index = solution.Value(routing.NextVar(index))
        # Ajouter le retour au dépôt
        route.append(tous_les_points[manager.IndexToNode(index)])
        
        # On ne retient que les véhicules ayant au moins une livraison (départ -> livraison(s) -> retour)
        if len(route) > 2:
            current_routes[vehicle_id] = route

def calc_route_distance(route):
    """Calcule la distance totale d'un itinéraire en mètres"""
    dist = 0
    for i in range(len(route)-1):
        dist += geodesic(route[i], route[i+1]).meters
    return dist

best_vehicle = None
best_extra_dist = float('inf')
best_new_route = None

# 2. Chercher la meilleure insertion pour chaque véhicule actif
# (On suppose ici que l'on peut insérer la commande n'importe où dans le trajet restant,
# pour simplifier, on teste toutes les combinaisons d'insertion i et j avec i <= j)
for vid, route in current_routes.items():
    original_dist = calc_route_distance(route)
    
    # On teste l'insertion du pickup à l'index i et du dropoff à l'index j
    # (i et j entre 1 et len(route)-1 pour ne pas remplacer le dépôt de départ/arrivée)
    for i in range(1, len(route)):
        for j in range(i, len(route)):
            # Créer un itinéraire candidat
            new_route = route[:i] + [new_pickup] + route[i:j] + [new_dropoff] + route[j:]
            new_dist = calc_route_distance(new_route)
            extra_dist = new_dist - original_dist
            
            # Si cette insertion est la plus courte trouvée, on la retient
            if extra_dist < best_extra_dist:
                best_extra_dist = extra_dist
                best_vehicle = vid
                best_new_route = new_route

# 3. Afficher les résultats et la carte de la nouvelle tournée
if best_vehicle is not None:
    print(f"-> Le véhicule {best_vehicle + 1} est le mieux placé pour prendre en charge cette course.")
    print(f"-> Distance supplémentaire engendrée : {best_extra_dist/1000:.2f} km\n")
    
    # Génération des liens Google Maps
    def generate_gmaps_link(route):
        url_parts = ["https://www.google.com/maps/dir"]
        for lat, lon in route:
            url_parts.append(f"{lat},{lon}")
        return "/".join(url_parts)
    
    old_gmaps_url = generate_gmaps_link(current_routes[best_vehicle])
    new_gmaps_url = generate_gmaps_link(best_new_route)
    
    print("=== LIENS GPS POUR LE CHAUFFEUR ===")
    print(f"Ancien itinéraire GPS :\n{old_gmaps_url}\n")
    print(f"Nouveau itinéraire GPS :\n{new_gmaps_url}\n")
    
    # Visualisation de la nouvelle tournée mise à jour
    map_sim = folium.Map(location=[36.8065, 10.1815], zoom_start=11)
    
    # Marqueur Entrepôt
    folium.Marker(
        ENTREPOT, 
        popup="<b>Entrepôt</b>", 
        icon=folium.Icon(color='red', icon='home')
    ).add_to(map_sim)
    
    # Marqueurs de la nouvelle demande
    folium.Marker(
        new_pickup, 
        popup="<b>Nouveau Pickup</b>", 
        icon=folium.Icon(color='green', icon='arrow-up')
    ).add_to(map_sim)
    
    folium.Marker(
        new_dropoff, 
        popup="<b>Nouvelle Destination</b>", 
        icon=folium.Icon(color='purple', icon='arrow-down')
    ).add_to(map_sim)
    
    # Tracer l'ancien itinéraire pour ce véhicule
    folium.PolyLine(
        current_routes[best_vehicle],
        color='gray',
        weight=4,
        opacity=0.6,
        dash_array='5, 5',
        tooltip=f"Ancienne Tournée - Chauffeur {best_vehicle + 1}"
    ).add_to(map_sim)
    
    # Tracer le nouvel itinéraire pour ce véhicule
    folium.PolyLine(
        best_new_route,
        color='blue',
        weight=4,
        opacity=0.8,
        tooltip=f"Tournée mise à jour - Chauffeur {best_vehicle + 1}"
    ).add_to(map_sim)
    
    display(map_sim)
else:
    print("Aucun véhicule actif disponible pour simuler l'insertion.")

=== NOUVELLE DEMANDE EN TEMPS RÉEL ===
Pickup      : (36.84, 10.22)
Destination : (36.86, 10.25)

-> Le véhicule 5 est le mieux placé pour prendre en charge cette course.
-> Distance supplémentaire engendrée : 3.31 km

=== LIENS GPS POUR LE CHAUFFEUR ===
Ancien itinéraire GPS :
https://www.google.com/maps/dir/36.82857,10.20616/36.7977845733307,10.208029773399304/36.7894184,10.1726889/36.7509827803322,10.223136940868464/36.71009442156114,10.187665025735477/36.69405351941644,10.121403347998315/36.66236058879193,10.09334818416276/36.630959670383966,10.0795068318421/36.62541696451701,10.10604904440124/36.54348601624928,10.191773802229878/36.66026213054205,10.292591562563905/36.69853002528086,10.248855276393057/36.69787706280917,10.299370075187936/36.67620158257303,10.351971160849232/36.65205442935984,10.399092473308029/36.67326660158319,10.395448725187949/36.72393219161856,10.315360484226224/36.7678233987053,10.272401372268078/36.78457856578412,10.246878031113509/36.84952823967741,10.28555

: 